In [1]:
import pandas as pd
import requests
import time
import pickle
import numpy as np
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

In [2]:
#1. CONFIGURATION
API_KEY = 'YOUR_API_KEY_HERE'
BASE_URL = "https://api.nasa.gov/neo/rest/v1/feed"

In [3]:
#2. DEFINE DATA FETCHING FUNCTIONS
def fetch_asteroid_data(start_date, end_date):
    """Fetches asteroid data for a specific 7-day window."""
    params = {
        'start_date': start_date,
        'end_date': end_date,
        'api_key': API_KEY
    }
    response = requests.get(BASE_URL, params=params)
    
    if response.status_code != 200:
        return []

    data = response.json()
    near_earth_objects = data.get('near_earth_objects', {})
    asteroid_list = []

    for date, asteroids in near_earth_objects.items():
        for asteroid in asteroids:
            try:
                # Extract features safely
                close_approach = asteroid['close_approach_data'][0]
                velocity = float(close_approach['relative_velocity']['kilometers_per_hour'])
                miss_distance = float(close_approach['miss_distance']['kilometers'])
                magnitude = float(asteroid['absolute_magnitude_h'])
                
                # Diameter comes in a range, we'll use the max estimate
                diameter_max = asteroid['estimated_diameter']['kilometers']['estimated_diameter_max']
                diameter_min = asteroid['estimated_diameter']['kilometers']['estimated_diameter_min']
                
                is_hazardous = asteroid['is_potentially_hazardous_asteroid']

                asteroid_list.append({
                    'absolute_magnitude': magnitude,
                    'est_diameter_min': diameter_min,
                    'est_diameter_max': diameter_max,
                    'relative_velocity': velocity,
                    'miss_distance': miss_distance,
                    'is_hazardous': is_hazardous
                })
            except (KeyError, IndexError):
                continue # Skip malformed records
            
    return asteroid_list

In [4]:
def get_big_dataset():
    """Loops over 8 weeks to build a larger dataset."""
    all_asteroids = []
    start_date = datetime.strptime("2023-01-01", "%Y-%m-%d")
    
    print("--- Starting Data Collection (This may take ~10-20 seconds) ---")
    
    # Fetch 8 weeks of data
    for i in range(8): 
        end_date = start_date + timedelta(days=7)
        s_str = start_date.strftime("%Y-%m-%d")
        e_str = end_date.strftime("%Y-%m-%d")
        
        print(f"Fetching Week {i+1}: {s_str} to {e_str}...")
        week_data = fetch_asteroid_data(s_str, e_str) 
        all_asteroids.extend(week_data)
        
        start_date = end_date + timedelta(days=1)
        time.sleep(1) # Be nice to the API

    return pd.DataFrame(all_asteroids)

#EXECUTE DATA COLLECTION
df = get_big_dataset()

print(f"\nTotal Asteroids Collected: {len(df)}")
print(f"Hazardous Count: {df['is_hazardous'].sum()}")

# Save raw data just in case
df.to_csv("nasa_asteroids_big.csv", index=False)

--- Starting Data Collection (This may take ~10-20 seconds) ---
Fetching Week 1: 2023-01-01 to 2023-01-08...
Fetching Week 2: 2023-01-09 to 2023-01-16...
Fetching Week 3: 2023-01-17 to 2023-01-24...
Fetching Week 4: 2023-01-25 to 2023-02-01...
Fetching Week 5: 2023-02-02 to 2023-02-09...
Fetching Week 6: 2023-02-10 to 2023-02-17...
Fetching Week 7: 2023-02-18 to 2023-02-25...
Fetching Week 8: 2023-02-26 to 2023-03-05...

Total Asteroids Collected: 1095
Hazardous Count: 55


In [5]:
df.head()

,absolute_magnitude,est_diameter_min,est_diameter_max,relative_velocity,miss_distance,is_hazardous
0,19.51,0.333085,0.744801,20959.820766,3.956596e+07,False
1,18.39,0.557898,1.247498,58249.689663,3.933082e+07,False
2,25.30,0.023150,0.051765,24703.743910,8.526777e+06,False
3,24.70,0.030518,0.068240,25881.197586,6.497596e+07,False
4,20.60,0.201630,0.450858,86987.581071,1.538437e+07,True


In [6]:
df.shape

(1095, 6)

In [7]:
df.isnull().sum()

absolute_magnitude    0
est_diameter_min      0
est_diameter_max      0
relative_velocity     0
miss_distance         0
is_hazardous          0
dtype: int64

In [8]:
df["is_hazardous"].value_counts()

is_hazardous
False    1040
True       55
Name: count, dtype: int64

In [9]:
#4. PREPARE FOR TRAINING
if len(df) == 0:
    print("Error: No data collected. Check your API Key.")
else:
    # Separate Features (X) and Target (y)
    X = df.drop(['is_hazardous'], axis=1)
    y = df['is_hazardous']

    #5. HANDLE IMBALANCE (SMOTE)
    # We check how many hazardous asteroids we actually found
    hazardous_count = y.sum()
    
    # We adjust k_neighbors dynamically. 
    # If we found 3 hazardous asteroids, neighbors must be < 3 (so 2).
    # If we found 50, we can use the default 5.
    k = 1 if hazardous_count <= 2 else (2 if hazardous_count <= 5 else 5)
    
    print(f"\nApplying SMOTE (Synthetic Minority Over-sampling) with k_neighbors={k}...")
    smote = SMOTE(random_state=42, k_neighbors=k)
    X_resampled, y_resampled = smote.fit_resample(X, y)
    
    print(f"Original shape: {y.value_counts().to_dict()}")
    print(f"Resampled shape: {y_resampled.value_counts().to_dict()}")

    #6. TRAIN MODEL
    print("\nTraining Random Forest Model...")
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model.fit(X_resampled, y_resampled)
    print("Training Complete!")

    #7. SAVE MODEL (PICKLE)
    filename = 'asteroid_hazard_model.pkl'
    with open(filename, 'wb') as file:
        pickle.dump(rf_model, file)
    
    print(f"SUCCESS: Model saved to '{filename}'")

    #8. VERIFY (TEST LOAD)
    print("\n--- Verifying Saved Model ---")
    with open(filename, 'rb') as file:
        loaded_model = pickle.load(file)
        
    # Fake asteroid for testing
    test_asteroid = np.array([[20.5, 0.2, 0.45, 55000, 1200000]])
    pred = loaded_model.predict(test_asteroid)
    print(f"Test Prediction for random rock: {'HAZARDOUS' if pred[0] else 'Safe'}")


Applying SMOTE (Synthetic Minority Over-sampling) with k_neighbors=5...
Original shape: {False: 1040, True: 55}
Resampled shape: {False: 1040, True: 1040}

Training Random Forest Model...
Training Complete!
SUCCESS: Model saved to 'asteroid_hazard_model.pkl'

--- Verifying Saved Model ---
Test Prediction for random rock: HAZARDOUS


C:\Users\shiva\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
